# SRT-M15：手机 Colab 增量学习

独立于旧 Notebook 的新学习入口。通过 GitHub `main/SRT-M15` 获取**最新代码**，先训练或恢复问答权重，再用**人工核实**的问答进行增量训练。训练失败时不会覆盖正式权重。

注意：模型仍是小规模实验网络；未参与训练的问句必须表现出真实进步，才会接受新权重。

In [ ]:
# 第 1 步：下载并核对 GitHub 最新版本（不使用 DeepSeek/API Key）
from urllib.request import Request, urlopen
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import io, shutil
URL = "https://github.com/mkusidodohsvebdks888/x/archive/refs/heads/main.zip"
with urlopen(Request(URL, headers={"User-Agent":"SRT-M15-Colab"}), timeout=120) as response:
    archive_bytes = response.read()
project = Path("/content/SRT-M15")
project.mkdir(parents=True, exist_ok=True)
with ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        parts = PurePosixPath(member.filename).parts
        if len(parts) < 3 or parts[1] != "SRT-M15" or member.is_dir():
            continue
        relative = PurePosixPath(*parts[2:])
        if any(part in (".", "..") for part in relative.parts):
            raise ValueError("unsafe zip entry")
        destination = project.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
print("源代码目录:",project)
print("增量训练器存在:",(project/"continual_qa.py").is_file())
%cd /content/SRT-M15


In [ ]:
# 第 2 步：先运行离线单元测试
!python -m unittest discover -s tests -p test_learning_pipeline.py -v


In [ ]:
# 第 3 步：如果没有已有的监督问答检查点，先训练基础模型。
from pathlib import Path
if not Path("checkpoints/srt_m15_qa.pt").exists():
    print("首次训练：")
    !python train_qa.py --steps 1600 --device auto --output checkpoints/srt_m15_qa.pt
else:
    print("已有监督问答检查点；跳过重复基础训练。")


In [ ]:
# 第 4 步：复制审核样本模板，手动编辑其中的问题、答案和来源。
from pathlib import Path
source = Path("learning_data/verified_qa.example.jsonl")
approved = Path("learning_data/verified_qa.jsonl")
if not approved.exists():
    approved.write_text(source.read_text(encoding="utf-8"), encoding="utf-8")
print("编辑此文件后再训练：", approved)
print(approved.read_text(encoding="utf-8")[:1000])


In [ ]:
# 第 5 步：带旧任务回放、未见问句测试、防遗忘门槛的增量训练
!python continual_qa.py --data learning_data/verified_qa.jsonl --steps 300 --eval-every 50 --device auto


In [ ]:
# 第 6 步：查看评估报告。只有 promoted=true 才创建/替换增量权重。
from pathlib import Path
result = Path("checkpoints/learning_report.json")
print(result.read_text(encoding="utf-8")[:5000] if result.exists() else "未生成学习报告")


In [ ]:
# 第 7 步：备份已经通过门槛的检查点和训练报告，不导出私有数据。
from google.colab import files
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
out = Path("/content/SRT-M15-Verified-Results.zip")
names = ("checkpoints/srt_m15_qa.pt", "checkpoints/srt_m15_qa_continual.pt",
         "checkpoints/learning_report.json")
with ZipFile(out, "w", ZIP_DEFLATED) as archive:
    for name in names:
        path = Path(name)
        if path.is_file():
            archive.write(path, arcname=name)
files.download(str(out))
